# Playground

Play with existing code

## `ingest.py` v1:

In [1]:
import os
import getpass

In [2]:
#if not os.environ.get("HF_TOKEN"):
#  os.environ["HF_TOKEN"] = getpass.getpass("insert HF_TOKEN:")

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from ingestion.chunker import Chunker
from ingestion.embedder import Embedder
from ingestion.vector_store import VectorStore

c:\dev\projects\buju_banton\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
chunker  = Chunker(chunk_size=500, chunk_overlap=50)

In [6]:
embedder = Embedder(model_name="all-MiniLM-L6-v2")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5123.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dimension: 384


### 1. Chunk

In [7]:
print("Chunking documents...")
docs = chunker.chunk_directory("./data")
print(f"Total chunks: {len(docs)}\n")

Chunking documents...
  Loading: sample.pdf ... 85 chunks
  Loading: sample.txt ... 88 chunks
Total chunks: 173



In [8]:
type(docs), len(docs)

(list, 173)

In [9]:
somedoc = docs[0]

In [10]:
print(f"chunk_index: {somedoc.chunk_index}\n")
print(f"content: {somedoc.content}\n")
print(f"metadata: {somedoc.metadata}\n")
print(f"page: {somedoc.page}\n")
print(f"source: {somedoc.source}\n")

chunk_index: 0

content: Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@google.com noam@google.com nikip@google.com usz@google.com
LlionJones∗ AidanN.Gomez∗ † ŁukaszKaiser∗
GoogleResearch UniversityofToronto GoogleBrain

metadata: {'source': 'data\\raw\\pdf\\sample.pdf', 'total_pages': 15, 'chunk_index': 0}

page: None

source: data\raw\pdf\sample.pdf



### 2. Embed

In [11]:
print("Embedding chunks...")
embeddings = embedder.embed_documents(docs)

Embedding chunks...


Batches: 100%|██████████| 6/6 [00:04<00:00,  1.31it/s]


In [12]:
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Sample vector (first 5 dims): {embeddings[0][:5]}")

Embedding matrix shape: (173, 384)
Sample vector (first 5 dims): [-0.08184301 -0.00166705  0.04545735 -0.0561004  -0.04109712]


### 3. Index

In [13]:
print("\nBuilding vector store...")
store = VectorStore(dimension=embedder.dimension)
store.add(docs, embeddings)


Building vector store...
  Indexed 173 chunks — total: 173


### 4. Persist

In [14]:
store.save("./vector_store")
print("Done.")

Saved 173 vectors → vector_store
Done.
